# SmartPayEnv Theme-4 Judge Repro (Colab, Self-Contained, Unsloth + TRL@git)

Self-contained notebook. Does NOT import anything from the repo.

Pipeline:
1. Install deps (Unsloth + TRL from GitHub)
2. HF login (uses your HF credits)
3. Connect to deployed SmartPayEnv Space
4. Collect group-relative preference pairs (inline)
5. Baseline eval (random + heuristic) on frozen seed
6. Train policy with Unsloth FastLanguageModel + TRL DPO
7. Trained-policy eval on the same frozen seed
8. Plots:
   - rollout reward curve
   - DPO training loss
   - before/after mean reward (random vs heuristic vs trained)
   - mean reward per risk bucket (low / medium / high)
9. Save artifacts to ./artifacts

Hackathon: OpenEnv (India 2026), Theme #4 — Self-Improvement.
Space: https://huggingface.co/spaces/Pratap-K/SmartPayEnv

## 1. Install dependencies (Unsloth + TRL from GitHub)

In [ ]:
!pip -q install --upgrade pip
!pip -q install "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip -q install "trl @ git+https://github.com/huggingface/trl.git"
!pip -q install --upgrade transformers accelerate peft bitsandbytes datasets huggingface_hub matplotlib pandas requests numpy

## 2. Authenticate Hugging Face

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Configuration

In [ ]:
import os, json, random
import numpy as np

QUICK_MODE = True
ENV_URL = 'https://pratap-k-smartpayenv.hf.space'
DIFFICULTY = 2
SEED = 42

ROLLOUT_STEPS = 60 if QUICK_MODE else 240
GROUP_SIZE = 6 if QUICK_MODE else 10
EVAL_EPISODES = 3 if QUICK_MODE else 5
EVAL_STEPS_PER_EPISODE = 30 if QUICK_MODE else 60

MODEL_ID = 'unsloth/Qwen2.5-0.5B-Instruct'
MAX_SEQ_LEN = 2048
LOAD_IN_4BIT = True

os.makedirs('artifacts', exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
print('Config ready. QUICK_MODE =', QUICK_MODE, '| MODEL_ID =', MODEL_ID)

## 4. Health check

In [ ]:
import requests
r = requests.get(f'{ENV_URL}/health', timeout=30)
print('Health:', r.status_code, r.text[:120])

## 5. Inline env helpers

In [ ]:
def env_reset(difficulty=DIFFICULTY):
    res = requests.post(f'{ENV_URL}/reset', json={'difficulty': int(difficulty)}, timeout=30)
    res.raise_for_status()
    payload = res.json()
    return payload.get('observation', payload)

def env_step(action):
    res = requests.post(f'{ENV_URL}/step', json={'action': action}, timeout=30)
    res.raise_for_status()
    return res.json()

def env_simulate(action):
    res = requests.post(f'{ENV_URL}/simulate', json={'action': action}, timeout=30)
    res.raise_for_status()
    return res.json()

def all_actions():
    out = []
    for g in (0,1,2):
        for f in (0,1,2,3):
            for r in (0,1):
                out.append({'gateway': g, 'fraud_decision': f, 'retry_strategy': r})
    return out

ACTIONS = all_actions()
print('Total candidate actions:', len(ACTIONS))

## 6. Collect group-relative preference pairs

In [ ]:
def collect_pairs(steps=ROLLOUT_STEPS, group=GROUP_SIZE, difficulty=DIFFICULTY):
    obs = env_reset(difficulty)
    pairs, reward_trace = [], []
    for _ in range(steps):
        sampled = random.sample(ACTIONS, k=min(group, len(ACTIONS)))
        scored = []
        for a in sampled:
            try:
                sim = env_simulate(a)
                scored.append((a, float(sim.get('reward', 0.0))))
            except requests.RequestException:
                continue
        if len(scored) < 2:
            break
        scored.sort(key=lambda x: x[1], reverse=True)
        best, best_r = scored[0]
        worst, worst_r = scored[-1]

        prompt = (
            'SmartPayEnv observation:\n'
            f'{json.dumps(obs, sort_keys=True)}\n'
            'Return one action JSON with fields: gateway, fraud_decision, retry_strategy.'
        )
        pairs.append({
            'prompt': prompt,
            'chosen': json.dumps(best, sort_keys=True),
            'rejected': json.dumps(worst, sort_keys=True),
            'chosen_reward': best_r,
            'rejected_reward': worst_r,
        })
        reward_trace.append(best_r)

        step_payload = env_step(best)
        obs = step_payload.get('observation', step_payload)
        if bool(obs.get('done', False)):
            obs = env_reset(difficulty)
    return pairs, reward_trace

pairs, rollout_rewards = collect_pairs()
print('Collected pairs:', len(pairs))

## 7. Baseline evaluation (random + heuristic) with risk-bucket breakdown

In [ ]:
def risk_bucket(obs):
    r = float(obs.get('observed_fraud_risk', 0.0))
    if r < 0.3:
        return 'low'
    if r < 0.65:
        return 'medium'
    return 'high'

def eval_policy(policy_fn, episodes=EVAL_EPISODES, steps=EVAL_STEPS_PER_EPISODE, difficulty=DIFFICULTY):
    all_rewards = []
    per_episode_means = []
    bucket_rewards = {'low': [], 'medium': [], 'high': []}
    for _ in range(episodes):
        obs = env_reset(difficulty)
        ep_rewards = []
        for _ in range(steps):
            bucket = risk_bucket(obs)
            action = policy_fn(obs)
            payload = env_step(action)
            obs = payload.get('observation', payload)
            r = float(obs.get('reward', payload.get('reward', 0.0)))
            ep_rewards.append(r)
            bucket_rewards[bucket].append(r)
            if bool(obs.get('done', False)):
                obs = env_reset(difficulty)
        all_rewards.extend(ep_rewards)
        per_episode_means.append(float(np.mean(ep_rewards)))
    bucket_means = {k: (float(np.mean(v)) if v else 0.0) for k, v in bucket_rewards.items()}
    return {
        'mean_reward': float(np.mean(all_rewards)) if all_rewards else 0.0,
        'per_episode_mean': per_episode_means,
        'bucket_means': bucket_means,
        'all_rewards': all_rewards,
    }

def random_policy(_obs):
    return random.choice(ACTIONS)

def heuristic_policy(obs):
    risk = float(obs.get('observed_fraud_risk', 0.0))
    rates = obs.get('gateway_success_rates', [0.9, 0.9, 0.9]) or [0.9, 0.9, 0.9]
    gateway = int(np.argmax(rates))
    if risk > 0.65:
        fd = 1
    elif risk > 0.4:
        fd = 2
    else:
        fd = 0
    return {'gateway': gateway, 'fraud_decision': fd, 'retry_strategy': 1}

baseline_random = eval_policy(random_policy)
baseline_heuristic = eval_policy(heuristic_policy)
print('Random baseline:', baseline_random['mean_reward'], baseline_random['bucket_means'])
print('Heuristic baseline:', baseline_heuristic['mean_reward'], baseline_heuristic['bucket_means'])

## 8. Train with Unsloth FastLanguageModel + TRL DPO

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import DPOConfig, DPOTrainer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = Dataset.from_list(pairs)
print(ds)

cfg = DPOConfig(
    output_dir='outputs/theme4_dpo_unsloth',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1 if QUICK_MODE else 2,
    logging_steps=2,
    learning_rate=5e-6,
    max_prompt_length=1024,
    max_length=1280,
    save_strategy='no',
    report_to=[],
    bf16=True,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=cfg,
    train_dataset=ds,
    processing_class=tokenizer,
)
trainer.train()

loss_history = [h.get('loss') for h in trainer.state.log_history if 'loss' in h]
print('Training loss points:', len(loss_history))

## 9. Trained-policy evaluation

In [ ]:
import re
import torch

FastLanguageModel.for_inference(model)
device = next(model.parameters()).device
ACTION_RE = re.compile(r'\{[^{}]*\}')

def parse_action(text):
    m = ACTION_RE.search(text)
    if not m:
        return {'gateway': 1, 'fraud_decision': 0, 'retry_strategy': 1}
    try:
        a = json.loads(m.group(0))
        return {
            'gateway': int(a.get('gateway', 1)) % 3,
            'fraud_decision': int(a.get('fraud_decision', 0)) % 4,
            'retry_strategy': int(a.get('retry_strategy', 1)) % 2,
        }
    except Exception:
        return {'gateway': 1, 'fraud_decision': 0, 'retry_strategy': 1}

def trained_policy(obs):
    prompt = (
        'SmartPayEnv observation:\n'
        f'{json.dumps(obs, sort_keys=True)}\n'
        'Return one action JSON with fields: gateway, fraud_decision, retry_strategy.'
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return parse_action(text)

trained_eval = eval_policy(trained_policy)
print('Trained policy mean reward:', trained_eval['mean_reward'])
print('Trained per-bucket:', trained_eval['bucket_means'])

## 10. Plots and saved artifacts

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.plot(rollout_rewards, label='Best-action reward per rollout step')
plt.xlabel('Rollout step')
plt.ylabel('Reward')
plt.title('Group-relative rollout reward (data-collection phase)')
plt.legend()
plt.tight_layout()
plt.savefig('artifacts/rollout_reward_curve.png', dpi=140)
plt.show()

if loss_history:
    plt.figure(figsize=(8,4))
    plt.plot(loss_history, label='DPO training loss')
    plt.xlabel('Logging step')
    plt.ylabel('Loss')
    plt.title('TRL DPO training loss (Unsloth)')
    plt.legend()
    plt.tight_layout()
    plt.savefig('artifacts/training_loss.png', dpi=140)
    plt.show()

labels = ['Random', 'Heuristic', 'Trained LLM']
values = [baseline_random['mean_reward'], baseline_heuristic['mean_reward'], trained_eval['mean_reward']]
plt.figure(figsize=(7,4))
bars = plt.bar(labels, values, color=['#bbb','#88c','#4a8'])
for b, v in zip(bars, values):
    plt.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}', ha='center')
plt.ylabel('Mean reward (frozen holdout)')
plt.title('Before vs After Training')
plt.tight_layout()
plt.savefig('artifacts/before_after_rewards.png', dpi=140)
plt.show()

buckets = ['low', 'medium', 'high']
rand_b = [baseline_random['bucket_means'][b] for b in buckets]
heur_b = [baseline_heuristic['bucket_means'][b] for b in buckets]
trnd_b = [trained_eval['bucket_means'][b] for b in buckets]
x = np.arange(len(buckets))
w = 0.27
plt.figure(figsize=(8,4))
plt.bar(x - w, rand_b, width=w, label='Random', color='#bbb')
plt.bar(x,     heur_b, width=w, label='Heuristic', color='#88c')
plt.bar(x + w, trnd_b, width=w, label='Trained LLM', color='#4a8')
plt.xticks(x, [b.title()+' Risk' for b in buckets])
plt.ylabel('Mean reward')
plt.title('Per Risk-Bucket Reward (frozen holdout)')
plt.legend()
plt.tight_layout()
plt.savefig('artifacts/per_bucket_rewards.png', dpi=140)
plt.show()

summary = {
    'env_url': ENV_URL,
    'model_id': MODEL_ID,
    'quick_mode': QUICK_MODE,
    'pairs_collected': len(pairs),
    'baseline_random_mean_reward': baseline_random['mean_reward'],
    'baseline_heuristic_mean_reward': baseline_heuristic['mean_reward'],
    'trained_mean_reward': trained_eval['mean_reward'],
    'reward_gain_vs_random': trained_eval['mean_reward'] - baseline_random['mean_reward'],
    'reward_gain_vs_heuristic': trained_eval['mean_reward'] - baseline_heuristic['mean_reward'],
    'per_bucket': {
        'random': baseline_random['bucket_means'],
        'heuristic': baseline_heuristic['bucket_means'],
        'trained': trained_eval['bucket_means'],
    },
    'rollout_reward_trace': rollout_rewards,
    'training_loss_history': loss_history,
    'eval_per_episode': {
        'random': baseline_random['per_episode_mean'],
        'heuristic': baseline_heuristic['per_episode_mean'],
        'trained': trained_eval['per_episode_mean'],
    },
}
with open('artifacts/run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(json.dumps({k:v for k,v in summary.items() if k not in ('rollout_reward_trace','training_loss_history')}, indent=2))

## 11. (Optional) Upload artifacts

In [ ]:
# !huggingface-cli upload <your-hf-repo> artifacts artifacts --repo-type dataset